In [ ]:
# ==============================================================================
# LOGICAL OBSERVATION IDENTIFIERS NAMES AND CODES (LOINC)
# 
# ARCHITECTURE NOTE: LOINC is a flat, multi-axial clinical catalog, not a 
# standard taxonomy. It contains mixed conceptual entities: complete clinical 
# observations, discrete "Parts" (LP), "Answers" (LA), and "Lists" (LL). 
#
# SSSOM ALIGNMENT STRATEGY: 
# To force this flat structure into a hierarchical CSV schema, we construct 
# a synthetic breadcrumb path based on entity type. For clinical survey items 
# (Observations), we extract the survey instrument name (e.g., FACIT, OMAHA) 
# to group related questions logically within the hierarchy path.
# ==============================================================================

import os
import sys
import time
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

load_dotenv(os.path.join(config_dir, ".env"))
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")
LOINC_USER = os.getenv("LOINC_USERNAME")
LOINC_PASS = os.getenv("LOINC_PASSWORD")

if not LOINC_USER or not LOINC_PASS:
    raise ValueError("CRITICAL: LOINC_USERNAME and LOINC_PASSWORD must be set in config/.env")

auth = HTTPBasicAuth(LOINC_USER, LOINC_PASS)

# --- 2. Registry Lookup ---
SOURCE_PREFIX = "LOINC"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="Logical Observation Identifiers Names and Codes",
    fallback_uri="https://loinc.org/"
)

SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (Research; mailto:{contact_email})',
    'Accept': 'application/fhir+json'
}

# --- 3. Search & Extraction Functions ---
SEARCH_TERMS = [
    "religious", "religion", "spiritual", "spirituality", 
    "church", "belief", "faith", "chaplain", "pastor", "clergy"
]
KNOWN_SEEDS = ["81364-2", "76510-7", "28351-5", "28213-7", "70697-8"]

def safe_fhir_request(url, params=None):
    """Executes a FHIR API request with retries."""
    for attempt in range(3):
        try:
            res = requests.get(url, headers=HEADERS, auth=auth, params=params, timeout=15)
            if res.status_code == 200: return res.json()
            elif res.status_code == 404: return None
            time.sleep(2)
        except Exception:
            time.sleep(2)
    return None

def search_loinc_terms(term):
    """Searches the FHIR ValueSet for matching LOINC codes."""
    url = f"https://fhir.loinc.org/ValueSet/$expand"
    data = safe_fhir_request(url, params={'url': 'http://loinc.org/vs', 'filter': term})
    if data and 'expansion' in data:
        return [item.get('code') for item in data['expansion'].get('contains', []) if item.get('system') == 'http://loinc.org']
    return []

def get_concept_details(code):
    """Extracts the axes, classification types, and metadata for a specific LOINC code."""
    url = f"https://fhir.loinc.org/CodeSystem/$lookup"
    data = safe_fhir_request(url, params={'system': 'http://loinc.org', 'code': code})
    if not data: return None

    # Identify the specific entity type to guide path creation
    if code.startswith('LA'): 
        concept_type = "skos:Concept"
        hierarchy_prefix = "LOINC Answer"
    elif code.startswith('LP'): 
        concept_type = "owl:Property"
        hierarchy_prefix = "LOINC Part"
    elif code.startswith('LL'): 
        concept_type = "skos:Collection"
        hierarchy_prefix = "LOINC Answer List"
    else: 
        concept_type = "owl:Class"
        hierarchy_prefix = "LOINC Observation"

    raw_fields = {
        'fallback_display': '', 'long_common_name': '', 'fully_specified_name': '',
        'component_display': '', 'survey_text': '', 'survey_src': '', 
        'survey_class': '', 'survey_copyright': '',
        'status': 'active', 'synonyms': [], 'has_translation': '', 'parent_id': ''
    }

    loinc_axes = {'component': '', 'property': '', 'time': '', 'system': '', 'scale': '', 'method': ''}

    for p in data.get('parameter', []):
        name = p.get('name')
        parts = p.get('part', [])
        
        if name == 'display':
            raw_fields['fallback_display'] = p.get('valueString', '')

        elif name == 'designation':
            val, use_code, use_display, lang = "", "", "", ""
            for part in parts:
                part_name = part.get('name')
                if part_name == 'value': val = part.get('valueString', '')
                elif part_name == 'use': 
                    use_code = part.get('valueCoding', {}).get('code', '')
                    use_display = part.get('valueCoding', {}).get('display', '')
                elif part_name == 'language': lang = part.get('valueCode', '').lower()
            
            if val:
                if not lang or lang.startswith('en'):
                    if use_code == 'FullySpecifiedName' or use_display == 'FullySpecifiedName': 
                        raw_fields['fully_specified_name'] = val
                    else: 
                        raw_fields['synonyms'].append(val)
                else:
                    raw_fields['has_translation'] = 'yes'

        elif name == 'property':
            prop_code, val_str, val_code = "", "", ""
            for part in parts:
                if part.get('name') == 'code': prop_code = part.get('valueCode', '')
                elif part.get('name') == 'value':
                    if 'valueString' in part: val_str = part.get('valueString', '')
                    elif 'valueCoding' in part:
                        val_str = part.get('valueCoding', {}).get('display', '')
                        val_code = part.get('valueCoding', {}).get('code', '')

            if prop_code == 'LONG_COMMON_NAME': raw_fields['long_common_name'] = val_str
            elif prop_code == 'COMPONENT':
                raw_fields['component_display'] = val_str
                loinc_axes['component'] = val_str
            elif prop_code == 'STATUS': raw_fields['status'] = val_str.lower()
            elif prop_code == 'parent': raw_fields['parent_id'] = val_code 
            
            # Extract Survey Metadata
            elif prop_code == 'SURVEY_QUEST_TEXT': raw_fields['survey_text'] = val_str
            elif prop_code == 'SURVEY_QUEST_SRC': raw_fields['survey_src'] = val_str
            elif prop_code == 'EXTERNAL_COPYRIGHT_LINK': raw_fields['survey_copyright'] = val_str
            elif prop_code == 'CLASS': raw_fields['survey_class'] = val_str
            
            # Map the remaining axes
            elif prop_code == 'PROPERTY': loinc_axes['property'] = val_str
            elif prop_code == 'TIME_ASPCT': loinc_axes['time'] = val_str
            elif prop_code == 'SYSTEM': loinc_axes['system'] = val_str
            elif prop_code == 'SCALE_TYP': loinc_axes['scale'] = val_str
            elif prop_code == 'METHOD_TYP': loinc_axes['method'] = val_str

    if concept_type == "owl:Class" and not raw_fields['fully_specified_name']:
        if loinc_axes['component'] and loinc_axes['property'] and loinc_axes['time'] and loinc_axes['system'] and loinc_axes['scale']:
            fsn_parts = [
                loinc_axes['component'], loinc_axes['property'], 
                loinc_axes['time'], loinc_axes['system'], loinc_axes['scale']
            ]
            if loinc_axes['method']: fsn_parts.append(loinc_axes['method'])
            raw_fields['fully_specified_name'] = ":".join(fsn_parts)

    primary_label = raw_fields['long_common_name'] if raw_fields['long_common_name'] else raw_fields['fallback_display']
    
    description = raw_fields['survey_text']
    if description and raw_fields['survey_src']: description += f" (Source: {raw_fields['survey_src']})"
    
    # --- UPGRADE: Synthetic Survey Pathing ---
    path_components = [hierarchy_prefix]
    
    if concept_type == "owl:Class":
        survey_instrument = ""
        # 1. Check Copyright string
        if raw_fields['survey_copyright']: 
            survey_instrument = f"{raw_fields['survey_copyright']} Survey"
        # 2. Fallback to Source string
        elif raw_fields['survey_src']: 
            # Clean up string (e.g., "OMAHA II.10" -> "OMAHA")
            survey_instrument = f"{raw_fields['survey_src'].split()[0]} Survey"
        # 3. Fallback to Class 
        elif raw_fields['survey_class'].startswith("SURVEY"):
            # Clean up string (e.g., "SURVEY.NURSE.OMAHA" -> "OMAHA")
            survey_instrument = f"{raw_fields['survey_class'].split('.')[-1]} Survey"
            
        if survey_instrument: path_components.append(survey_instrument)
        
        if raw_fields['component_display'] and raw_fields['component_display'] != primary_label:
            path_components.append(raw_fields['component_display'])
            
    path_components.append(primary_label)
    hierarchy_path = " > ".join([x for x in path_components if x])
    
    return {
        'Source_System': SOURCE_NAME,
        'Primary_Label': primary_label,
        'CURIE': f"{SOURCE_PREFIX}:{code}",
        'Formal_Label': raw_fields['fully_specified_name'],
        'Concept_Type': concept_type,
        'Hierarchy_Path': hierarchy_path,
        'Synonyms': " | ".join(list(set(raw_fields['synonyms']))),
        'Description': description,
        'Parent_IDs': raw_fields['parent_id'],
        'Concept_ID': str(code),
        'URI': f"{BASE_URI}{code}",
        'Has_Translation': raw_fields['has_translation'],
        'Status': raw_fields['status'],
        'Crosswalks': ""
    }

# --- 4. Persistent Progress Tracking & Schema Drift Check ---
print(f"[*] Starting {SOURCE_PREFIX} Discovery Phase...")

if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: Found {len(processed_ids)} LOINC concepts already saved.")
else:
    processed_ids = set()

all_target_ids = set(KNOWN_SEEDS)
for term in SEARCH_TERMS:
    print(f"    Searching LOINC for: '{term}'...")
    all_target_ids.update(search_loinc_terms(term))
    time.sleep(0.5) 

total_targets = len(all_target_ids)
print(f"\nDiscovery complete. Found {total_targets} unique LOINC concepts to process.")

# --- 5. Extraction Pipeline ---
for i, cid in enumerate(sorted(list(all_target_ids)), 1):
    if not cid or cid in processed_ids: continue
        
    print(f"\r[{i}/{total_targets}] Processing {cid}{' ' * 20}", end="", flush=True)
    
    extracted_data = get_concept_details(cid)
    if not extracted_data: continue

    clean_row = finalize_row(extracted_data)
    
    df_row = pd.DataFrame([clean_row])[COLUMN_ORDER]
    file_exists = os.path.isfile(raw_ingest_file)
    df_row.to_csv(raw_ingest_file, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
    
    processed_ids.add(cid)
    time.sleep(0.5)

print(f"\n\nIngest Complete! Saved {len(processed_ids)} records to {raw_ingest_file}")